# Population Delta Analysis for Odorant Receptors

**Goal:** Compare population-level receptor activation patterns at **low (10⁻⁸ M)** vs. **high (10⁻³ M)** odorant concentration across all odorants from [Mainland et al. 2015](https://doi.org/10.1038/nn.3922). Odorants that recruit many new receptors at high concentration have a large "population delta"; those that saturate their receptor panel early have a small delta.

We then pick two representative molecules — one with a large delta and one with a small delta — and visualize their receptor activation patterns side-by-side on virtual epithelium heatmaps.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
%matplotlib inline

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────
DATA_DIR = "data/"

dr = pd.read_csv(f"{DATA_DIR}/DR.tsv", sep="\t", quotechar='"')
odors = pd.read_csv(f"{DATA_DIR}/Odors.tsv", sep="\t")
receptors = pd.read_csv(
    f"{DATA_DIR}/Receptors.tsv", sep="\t", usecols=["OR", "Gene"]
)

# Build lookup dicts
odor_names = dict(zip(odors["Odor"], odors["OdorName"]))
receptor_genes = dict(zip(receptors["OR"], receptors["Gene"]))

# Drop vector control and rows with missing response values
receptors = receptors[~receptors["Gene"].str.contains("Vector", case=False, na=False)]
real_or_ids = set(receptors["OR"])
dr = dr[dr["OR"].isin(real_or_ids)]
dr = dr.dropna(subset=["NormalizedLuc"])

print(f"Loaded {len(dr)} dose-response measurements")
print(f"  {dr['OR'].nunique()} receptors, {dr['Odor'].nunique()} odorants")
print(f"  {dr.groupby(['OR', 'Odor']).ngroups} unique (receptor, odorant) pairs")

In [ ]:
# ── Hill / sigmoid model ───────────────────────────────────────────────
def hill_function(log_conc, baseline, top, log_ec50, hill_n):
    """4-parameter Hill equation in log-concentration space."""
    return baseline + (top - baseline) / (
        1.0 + 10.0 ** ((log_ec50 - log_conc) * hill_n)
    )


def fit_sigmoid(concentrations, responses):
    """
    Fit a Hill curve to a single (receptor, odorant) dose-response series.
    Returns (params, r_squared) or (None, None) on failure.
    """
    log_c = np.log10(concentrations)
    y = responses

    # Reasonable starting guesses
    baseline_guess = np.percentile(y, 10)
    top_guess = np.percentile(y, 90)
    log_ec50_guess = np.median(log_c)
    hill_n_guess = 1.0

    p0 = [baseline_guess, top_guess, log_ec50_guess, hill_n_guess]
    bounds = (
        [-5, -5, -15, 0.1],  # lower bounds
        [5, 50, 0, 10],      # upper bounds
    )

    try:
        popt, _ = curve_fit(
            hill_function, log_c, y, p0=p0,
            bounds=bounds, maxfev=5000, method="trf",
        )
        y_pred = hill_function(log_c, *popt)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return popt, r_squared
    except (RuntimeError, ValueError):
        return None, None

In [ ]:
# ── Fit all (receptor, odorant) pairs ──────────────────────────────────
print("Fitting Hill curves to all (receptor, odorant) pairs...")

results = []
grouped = dr.groupby(["OR", "Odor"])

for (or_id, odor_id), group in grouped:
    conc = group["concentration"].values
    resp = group["NormalizedLuc"].values

    if len(conc) < 4 or len(np.unique(conc)) < 3:
        continue

    popt, r2 = fit_sigmoid(conc, resp)
    if popt is not None:
        baseline, top, log_ec50, hill_n = popt
        dynamic_range = top - baseline
        results.append({
            "OR": or_id,
            "Odor": odor_id,
            "baseline": baseline,
            "top": top,
            "log_ec50": log_ec50,
            "ec50": 10 ** log_ec50,
            "hill_n": hill_n,
            "r_squared": r2,
            "dynamic_range": dynamic_range,
            "gene": receptor_genes.get(or_id, f"OR{or_id}"),
            "odor_name": odor_names.get(odor_id, f"Odor{odor_id}"),
        })

fits = pd.DataFrame(results)
print(f"Successfully fit {len(fits)} curves out of {grouped.ngroups} pairs")

In [ ]:
# ── Filter for good fits ───────────────────────────────────────────────
good_fits = fits[(fits["r_squared"] > 0.3) & (fits["dynamic_range"] > 0.3)].copy()
print(f"Good fits (R²>0.3, dynamic range>0.3): {len(good_fits)}")
print(f"Covering {good_fits['Odor'].nunique()} odorants and {good_fits['OR'].nunique()} receptors")
print(f"\nMedian EC50 across all good fits: {good_fits['ec50'].median():.2e} M")

In [ ]:
# ── Compute population delta for all odorants ──────────────────────────
LOW_CONC = 1e-8   # faint whiff
HIGH_CONC = 1e-3   # strong sniff
LOG_LOW = np.log10(LOW_CONC)
LOG_HIGH = np.log10(HIGH_CONC)


def fractional_activation(row, log_conc):
    """Compute fractional saturation for a single fit at a given log-concentration."""
    response = hill_function(log_conc, row["baseline"], row["top"],
                             row["log_ec50"], row["hill_n"])
    if row["dynamic_range"] > 0:
        frac = (response - row["baseline"]) / row["dynamic_range"]
        return np.clip(frac, 0, 1)
    return 0.0


delta_records = []

for odor_id, odor_group in good_fits.groupby("Odor"):
    n_fits = len(odor_group)
    if n_fits < 5:
        continue

    act_low = odor_group.apply(fractional_activation, axis=1, log_conc=LOG_LOW).values
    act_high = odor_group.apply(fractional_activation, axis=1, log_conc=LOG_HIGH).values

    abs_delta = np.abs(act_high - act_low)
    signed_delta = act_high - act_low

    delta_records.append({
        "Odor": odor_id,
        "odor_name": odor_names.get(odor_id, f"Odor{odor_id}"),
        "n_good_fits": n_fits,
        "mean_delta": abs_delta.mean(),
        "L1_delta": abs_delta.sum(),
        "n_active_low": (act_low > 0.05).sum(),
        "n_active_high": (act_high > 0.05).sum(),
        "n_newly_recruited": ((act_low < 0.05) & (act_high > 0.05)).sum(),
        "mean_act_low": act_low.mean(),
        "mean_act_high": act_high.mean(),
    })

deltas = pd.DataFrame(delta_records).sort_values("mean_delta", ascending=False)
deltas = deltas.reset_index(drop=True)

print(f"Computed population delta for {len(deltas)} odorants (≥5 good fits each)")
print(f"\nTop 10 by mean per-receptor |Δactivation|:")
print(deltas[["odor_name", "mean_delta", "n_good_fits", "n_active_high",
              "n_newly_recruited"]].head(10).to_string(index=False))
print(f"\nBottom 5:")
print(deltas[["odor_name", "mean_delta", "n_good_fits", "n_active_high",
              "n_newly_recruited"]].tail(5).to_string(index=False))

In [ ]:
# ── Bar chart: odorants ranked by population delta ─────────────────────
fig, ax = plt.subplots(figsize=(10, max(6, len(deltas) * 0.35)))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(deltas)))
bars = ax.barh(range(len(deltas)), deltas["mean_delta"].values,
               color=colors[::-1], edgecolor="white", linewidth=0.3)
ax.set_yticks(range(len(deltas)))
ax.set_yticklabels(deltas["odor_name"].values, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Mean Per-Receptor |Δ Activation|  (10⁻⁸ → 10⁻³ M)", fontsize=11)
ax.set_title("Population Delta: How Much Does Concentration Change the Receptor Code?",
             fontsize=13, fontweight="bold")

# Annotate fit counts on bars
for i, row in deltas.iterrows():
    ax.text(row["mean_delta"] + 0.005, i,
            f'n={row["n_good_fits"]}', va="center", fontsize=7, color="gray")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Select molecules A (high delta) and B (low delta) ──────────────────
# Non-trivial filter: ≥10 good fits AND ≥3 receptors active at high conc
candidates = deltas[(deltas["n_good_fits"] >= 10) & (deltas["n_active_high"] >= 3)]
print(f"Non-trivial candidates: {len(candidates)} odorants (≥10 fits, ≥3 active at high)")

mol_a = candidates.iloc[0]   # highest delta among candidates
mol_b = candidates.iloc[-1]  # lowest delta among candidates

print(f"\n── Molecule A (HIGH delta): {mol_a['odor_name']} ──")
print(f"   Mean |Δ|: {mol_a['mean_delta']:.3f}")
print(f"   Good fits: {mol_a['n_good_fits']}")
print(f"   Active at low: {mol_a['n_active_low']}  |  Active at high: {mol_a['n_active_high']}")
print(f"   Newly recruited: {mol_a['n_newly_recruited']}")

print(f"\n── Molecule B (LOW delta): {mol_b['odor_name']} ──")
print(f"   Mean |Δ|: {mol_b['mean_delta']:.3f}")
print(f"   Good fits: {mol_b['n_good_fits']}")
print(f"   Active at low: {mol_b['n_active_low']}  |  Active at high: {mol_b['n_active_high']}")
print(f"   Newly recruited: {mol_b['n_newly_recruited']}")

In [ ]:
# ── Side-by-side sigmoid fits (2 columns × 6 rows) ────────────────────
fig, axes = plt.subplots(6, 2, figsize=(14, 22))
fig.suptitle(
    f"Dose-Response Curves — {mol_a['odor_name']} (high Δ) vs {mol_b['odor_name']} (low Δ)",
    fontsize=14, fontweight="bold", y=0.995,
)

for col_idx, (mol, label) in enumerate([(mol_a, "HIGH Δ"), (mol_b, "LOW Δ")]):
    odor_id = mol["Odor"]
    odor_name = mol["odor_name"]
    mol_fits = good_fits[good_fits["Odor"] == odor_id]

    # Pick top receptors by dynamic range with plausible EC50
    plausible = mol_fits[(mol_fits["ec50"] < 0.1) & (mol_fits["ec50"] > 1e-12)]
    if len(plausible) < 6:
        plausible = mol_fits.nlargest(6, "dynamic_range")
    else:
        plausible = plausible.nlargest(6, "dynamic_range")

    for row_idx, (_, row) in enumerate(plausible.iterrows()):
        if row_idx >= 6:
            break
        ax = axes[row_idx, col_idx]

        # Raw data
        mask = (dr["OR"] == row["OR"]) & (dr["Odor"] == odor_id)
        raw = dr[mask]
        ax.scatter(raw["concentration"], raw["NormalizedLuc"],
                   alpha=0.5, s=30, color="steelblue", zorder=3)

        # Fitted curve
        x_fit = np.logspace(-14, -1, 200)
        y_fit = hill_function(np.log10(x_fit), row["baseline"], row["top"],
                              row["log_ec50"], row["hill_n"])
        ax.plot(x_fit, y_fit, "r-", linewidth=2)

        # Mark low and high concentrations
        ax.axvline(LOW_CONC, color="green", linestyle=":", alpha=0.7,
                   linewidth=1.5, label="low" if row_idx == 0 else None)
        ax.axvline(HIGH_CONC, color="orange", linestyle="--", alpha=0.7,
                   linewidth=1.5, label="high" if row_idx == 0 else None)
        # Shade the delta region
        ax.axvspan(LOW_CONC, HIGH_CONC, alpha=0.05, color="yellow")

        ax.set_xscale("log")
        ax.set_xlabel("Concentration (M)", fontsize=8)
        ax.set_ylabel("Norm. Response", fontsize=8)
        ax.set_title(f"{row['gene']}  EC₅₀={row['ec50']:.1e}  R²={row['r_squared']:.2f}",
                     fontsize=9)
        if row_idx == 0:
            ax.legend(fontsize=7, loc="upper left")

    # Label the column
    axes[0, col_idx].annotate(
        f"{odor_name}\n({label})",
        xy=(0.5, 1.35), xycoords="axes fraction",
        ha="center", fontsize=12, fontweight="bold",
    )

    # Blank unused rows
    for row_idx in range(len(plausible), 6):
        axes[row_idx, col_idx].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
# ── Helper functions for activation vectors & heatmap rendering ────────

def compute_activation_vector(odor_id, log_conc, good_fits, real_or_ids, receptor_genes):
    """
    Compute fractional activation for ALL receptors at a given concentration
    for a specific odorant. Receptors without fits are treated as silent.
    Returns a pd.Series indexed by gene name.
    """
    odor_fits = good_fits[good_fits["Odor"] == odor_id]
    receptor_activation = {}

    for or_id in real_or_ids:
        gene = receptor_genes.get(or_id, f"OR{or_id}")
        match = odor_fits[odor_fits["OR"] == or_id]
        if len(match) > 0:
            row = match.iloc[0]
            response = hill_function(log_conc, row["baseline"], row["top"],
                                     row["log_ec50"], row["hill_n"])
            if row["dynamic_range"] > 0:
                frac = (response - row["baseline"]) / row["dynamic_range"]
                frac = np.clip(frac, 0, 1)
            else:
                frac = 0.0
            receptor_activation[gene] = frac
        else:
            receptor_activation[gene] = 0.0

    return pd.Series(receptor_activation)


def plot_epithelium(ax, activation, title, vmin=0, vmax=1, show_labels=True):
    """
    Render a virtual epithelium heatmap on the given axes.
    `activation` is a pd.Series indexed by gene name (already in desired order).
    """
    n = len(activation)
    n_cols = int(np.ceil(np.sqrt(n * 1.2)))
    n_rows = int(np.ceil(n / n_cols))

    values = activation.values.copy()
    labels = activation.index.values.copy()
    pad_size = n_rows * n_cols - n
    if pad_size > 0:
        values = np.concatenate([values, np.full(pad_size, np.nan)])
        labels = np.concatenate([labels, np.array([""] * pad_size)])

    grid = values.reshape(n_rows, n_cols)
    label_grid = labels.reshape(n_rows, n_cols)

    # Custom colormap
    cmap = mcolors.LinearSegmentedColormap.from_list(
        "receptor_activation",
        [
            (0.0, "#1a1a2e"),
            (0.01, "#16213e"),
            (0.1, "#0f3460"),
            (0.3, "#e94560"),
            (0.6, "#ff6b35"),
            (1.0, "#ffd700"),
        ],
    )
    cmap.set_bad(color="#2a2a3e")

    ax.set_facecolor("#0e0e1a")
    im = ax.imshow(grid, cmap=cmap, vmin=vmin, vmax=vmax,
                   aspect="equal", interpolation="nearest")

    # Grid lines
    for i in range(n_rows + 1):
        ax.axhline(i - 0.5, color="#2a2a3e", linewidth=0.3)
    for j in range(n_cols + 1):
        ax.axvline(j - 0.5, color="#2a2a3e", linewidth=0.3)

    # Labels on activated receptors
    if show_labels:
        for i in range(n_rows):
            for j in range(n_cols):
                val = grid[i, j]
                if np.isnan(val):
                    continue
                if val > 0.08:
                    gene = label_grid[i, j]
                    short = gene.replace("OR", "").split(" ")[0]
                    color = "black" if val > 0.5 else "white"
                    fontsize = 4.5 if val > 0.2 else 3.5
                    ax.text(j, i, short, ha="center", va="center",
                            fontsize=fontsize, color=color, fontweight="bold")

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, fontsize=10, fontweight="bold", color="white", pad=8)

    return im

print("Helper functions defined.")

In [ ]:
# ── Four virtual epithelium heatmaps (2×2 grid) ───────────────────────
#  Same receptor-to-position mapping across all panels (same shuffle seed)

# Compute activation vectors
act_a_low = compute_activation_vector(mol_a["Odor"], LOG_LOW, good_fits, real_or_ids, receptor_genes)
act_a_high = compute_activation_vector(mol_a["Odor"], LOG_HIGH, good_fits, real_or_ids, receptor_genes)
act_b_low = compute_activation_vector(mol_b["Odor"], LOG_LOW, good_fits, real_or_ids, receptor_genes)
act_b_high = compute_activation_vector(mol_b["Odor"], LOG_HIGH, good_fits, real_or_ids, receptor_genes)

# Use the SAME random shuffle for all panels
np.random.seed(42)
shared_order = np.random.permutation(act_a_low.index)

act_a_low = act_a_low.loc[shared_order]
act_a_high = act_a_high.loc[shared_order]
act_b_low = act_b_low.loc[shared_order]
act_b_high = act_b_high.loc[shared_order]

# Plot
fig, axes = plt.subplots(2, 2, figsize=(24, 20))
fig.patch.set_facecolor("#0e0e1a")

panels = [
    (axes[0, 0], act_a_low,  f"{mol_a['odor_name']} — LOW (10⁻⁸ M)"),
    (axes[0, 1], act_a_high, f"{mol_a['odor_name']} — HIGH (10⁻³ M)"),
    (axes[1, 0], act_b_low,  f"{mol_b['odor_name']} — LOW (10⁻⁸ M)"),
    (axes[1, 1], act_b_high, f"{mol_b['odor_name']} — HIGH (10⁻³ M)"),
]

for ax, activation, title in panels:
    im = plot_epithelium(ax, activation, title, vmin=0, vmax=1)

# Shared colorbar
cbar = fig.colorbar(im, ax=axes, shrink=0.4, pad=0.02, aspect=30)
cbar.set_label("Fractional Receptor Saturation", fontsize=12, color="white")
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(["0%", "25%", "50%", "75%", "100%"])
cbar.ax.tick_params(colors="white")
cbar.outline.set_edgecolor("#444")

fig.suptitle(
    "Virtual Epithelium: Low vs High Concentration\n"
    f"{mol_a['odor_name']} (high Δ) vs {mol_b['odor_name']} (low Δ)",
    fontsize=16, fontweight="bold", color="white", y=1.01,
)

plt.tight_layout()
plt.show()

# Stats summary
for label, low, high, mol in [("A", act_a_low, act_a_high, mol_a),
                               ("B", act_b_low, act_b_high, mol_b)]:
    print(f"\nMolecule {label} ({mol['odor_name']}):")
    print(f"  Active at low:  {(low > 0.05).sum()}")
    print(f"  Active at high: {(high > 0.05).sum()}")
    print(f"  Mean activation low:  {low.mean():.4f}")
    print(f"  Mean activation high: {high.mean():.4f}")

In [ ]:
# ── Difference heatmaps (1×2) ─────────────────────────────────────────
# Show activation_high - activation_low for A and B

diff_a = act_a_high - act_a_low
diff_b = act_b_high - act_b_low

# Diverging colormap: blue = decrease, white = no change, red = increase
div_cmap = mcolors.LinearSegmentedColormap.from_list(
    "delta_activation",
    [
        (0.0, "#2166ac"),   # blue: decreased
        (0.25, "#67a9cf"),
        (0.5, "#1a1a2e"),   # dark center: no change
        (0.75, "#ef8a62"),
        (1.0, "#b2182b"),   # red: increased
    ],
)


def plot_diff_epithelium(ax, diff_vec, title):
    """Render a difference heatmap on the given axes."""
    n = len(diff_vec)
    n_cols = int(np.ceil(np.sqrt(n * 1.2)))
    n_rows = int(np.ceil(n / n_cols))

    values = diff_vec.values.copy()
    labels = diff_vec.index.values.copy()
    pad_size = n_rows * n_cols - n
    if pad_size > 0:
        values = np.concatenate([values, np.full(pad_size, np.nan)])
        labels = np.concatenate([labels, np.array([""] * pad_size)])

    grid = values.reshape(n_rows, n_cols)
    label_grid = labels.reshape(n_rows, n_cols)

    cmap_local = div_cmap.copy()
    cmap_local.set_bad(color="#2a2a3e")

    # Symmetric limits so 0 is centered
    vabs = max(np.nanmax(np.abs(values)), 0.01)

    ax.set_facecolor("#0e0e1a")
    im = ax.imshow(grid, cmap=cmap_local, vmin=-vabs, vmax=vabs,
                   aspect="equal", interpolation="nearest")

    for i in range(n_rows + 1):
        ax.axhline(i - 0.5, color="#2a2a3e", linewidth=0.3)
    for j in range(n_cols + 1):
        ax.axvline(j - 0.5, color="#2a2a3e", linewidth=0.3)

    # Label receptors with large change
    for i in range(n_rows):
        for j in range(n_cols):
            val = grid[i, j]
            if np.isnan(val):
                continue
            if abs(val) > 0.1:
                gene = label_grid[i, j]
                short = gene.replace("OR", "").split(" ")[0]
                color = "white" if abs(val) > 0.3 else "#cccccc"
                ax.text(j, i, short, ha="center", va="center",
                        fontsize=4, color=color, fontweight="bold")

    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, fontsize=11, fontweight="bold", color="white", pad=8)
    return im, vabs


fig, axes = plt.subplots(1, 2, figsize=(24, 10))
fig.patch.set_facecolor("#0e0e1a")

im_a, vabs_a = plot_diff_epithelium(
    axes[0], diff_a,
    f"{mol_a['odor_name']} — Δ Activation (high − low)")
im_b, vabs_b = plot_diff_epithelium(
    axes[1], diff_b,
    f"{mol_b['odor_name']} — Δ Activation (high − low)")

# Use the larger vabs for both panels so they share the same scale
vabs_shared = max(vabs_a, vabs_b)
for ax_obj in axes:
    for im_obj in ax_obj.get_images():
        im_obj.set_clim(-vabs_shared, vabs_shared)

cbar = fig.colorbar(im_a, ax=axes, shrink=0.5, pad=0.02, aspect=30)
cbar.set_label("Δ Fractional Activation (high − low)", fontsize=12, color="white")
cbar.ax.tick_params(colors="white")
cbar.outline.set_edgecolor("#444")

fig.suptitle(
    "Concentration-Dependent Change in Receptor Activation",
    fontsize=15, fontweight="bold", color="white", y=1.02,
)

plt.tight_layout()
plt.show()

# Summary
print(f"\n{mol_a['odor_name']} (high Δ): {(np.abs(diff_a) > 0.1).sum()} receptors changed > 10%")
print(f"{mol_b['odor_name']} (low Δ):  {(np.abs(diff_b) > 0.1).sum()} receptors changed > 10%")